<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part3_transformers/09_clip_images_and_words/notebooks/09_clip_images_and_words_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 09: Images and Words: CLIP · Solutions

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a short comment. There is usually more than one good way to solve an exercise, and your numbers may differ a little from run to run, so read the comments as "what to look for" rather than exact results.

CLIP (Radford et al., 2021) was trained on about 400 million images paired with the text that
accompanied them on the web. It learned to place an image and a sentence that describes it close
together in one shared space. That opens up three things you could not do with the models of
Part 2: classify images without a single labeled example by writing down what each category looks
like, search a collection with a sentence, and build measurement scales whose ends are defined in
words. It also brings a problem from language models into vision: the wording of your prompt
changes your results.

In this notebook you will:

- put images and sentences into the same space and compare them
- classify your 40 coded images **zero-shot**, choosing the prompt wording without peeking at the
  test set, and compare with a linear probe on the same embeddings
- search the packaging collection with sentences, and with a mix of an image and words
- build a "healthy-looking" scale from two sentences and check it against the products' Nutri-Score
- audit how CLIP labels the occupations of members of Congress, men and women

**Data.** The protest test set (module 04), the Open Food Facts cereal packaging (module 02) and the
Congress portraits (module 01).

**Models.** `openai/clip-vit-base-patch32` (MIT license), and SigLIP 2
(`google/siglip2-base-patch16-224`, Apache 2.0) as a newer alternative.

**Runtime.** A GPU is recommended (*Runtime → Change runtime type → T4 GPU*), but the base CLIP
model also runs on a CPU for collections of a few hundred images.

## 0. Setup

The CLIP functions we write in section 1 are also in `course_torch.py`, so modules 10 to 12 can
reuse them.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from transformers import AutoModel, AutoProcessor
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

SEED = 42
np.random.seed(SEED)

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
BASE = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/"
for helper in ["course_data.py", "course_torch.py"]:
    if not Path(helper).exists():
        response = requests.get(BASE + helper, headers=HEADERS, timeout=30)
        response.raise_for_status()  # stop if the download failed, rather than saving an error page
        Path(helper).write_text(response.text)
import course_data
import course_torch
from course_torch import DEVICE, load_rgb

print("Running on:", DEVICE)

def show_grid(paths, labels=None, n_cols=8, size=1.6):
    """Show images in a grid, with an optional label under each."""
    paths = list(paths)
    labels = list(labels) if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        ax.imshow(load_rgb(path, 300))
        ax.set_title(str(label)[:28], fontsize=7)
    plt.tight_layout()
    plt.show()

train_table, test_table = course_data.protest_train_test()
packaging = course_data.get_packaging_corpus()
portraits = course_data.get_congress_portraits(n=None)

## 1. One space for images and words

CLIP has two encoders: a vision transformer for images (here ViT-B/32, the model with 32-pixel
patches from module 08) and a text transformer for captions. During training it saw batches of
32,768 image and caption pairs. For each image, the task was to pick its own caption out of all the
captions in the batch, and for each caption, its own image. This is **contrastive** learning: pull
the matching pairs together, push all the others apart. Nobody labeled anything; the captions did
the work.

The result is that both encoders map into the same 512-dimensional space. As in module 06, we
normalize every vector to length 1, so the dot product of an image vector and a sentence vector is
their cosine similarity.

In transformers 5, `get_image_features` returns an output object with the vector in
`pooler_output`, while older versions return the vector directly; the small `features` function
handles both.

In [ ]:
CLIP = "openai/clip-vit-base-patch32"
clip_processor = AutoProcessor.from_pretrained(CLIP)
clip_model = AutoModel.from_pretrained(CLIP).eval().to(DEVICE)

def features(output):
    return output if torch.is_tensor(output) else output.pooler_output

@torch.no_grad()
def embed_images(paths, batch_size=32):
    chunks = []
    for start in range(0, len(paths), batch_size):
        images = [load_rgb(p, 512) for p in paths[start:start + batch_size]]
        inputs = clip_processor(images=images, return_tensors="pt").to(DEVICE)
        chunks.append(features(clip_model.get_image_features(**inputs)).float().cpu())
    return torch.nn.functional.normalize(torch.cat(chunks), dim=1).numpy()

@torch.no_grad()
def embed_texts(texts):
    inputs = clip_processor(text=list(texts), return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    out = features(clip_model.get_text_features(**inputs)).float().cpu()
    return torch.nn.functional.normalize(out, dim=1).numpy()

examples = test_table["local_path"].head(6).tolist()
captions = ["a crowd of protesters holding signs", "people celebrating King's Day in orange",
            "a police officer", "a portrait of a politician", "a box of breakfast cereal", "an empty street"]
similarity = embed_images(examples) @ embed_texts(captions).T

fig, axes = plt.subplots(1, len(examples), figsize=(11, 2))
for ax, p in zip(axes, examples):
    ax.imshow(load_rgb(p, 300)); ax.axis("off")
plt.show()
plt.figure(figsize=(7, 3))
plt.imshow(similarity.T, cmap="viridis")
plt.yticks(range(len(captions)), captions, fontsize=8)
plt.xticks(range(len(examples)), [f"image {i + 1}" for i in range(len(examples))], fontsize=8)
plt.colorbar(label="cosine similarity")
plt.show()

The similarities are small numbers, often between 0.15 and 0.35, and differences of a few
hundredths matter. They only make sense **relative to each other**: comparing one image with
several sentences, or one sentence with several images. An absolute value like 0.27 doesn't mean
"27% protest".

## 2. Zero-shot classification

To classify without examples, write one sentence per category, embed the sentences, and give each
image the category whose sentence is most similar. CLIP's own paper turns the similarities into
probabilities by multiplying them by 100 (the model's learned "temperature") and applying a
softmax.

But which sentences? Below are five wordings for the same two categories. Choosing the wording
that works best **on the test set** would be cheating, just like tuning a classifier on the test
set: the score would be optimistic, and it would no longer measure how well the method works on
new images. So we compare wordings on the **training images**, whose Commons category labels are
imperfect but independent of the test set, and only then score the chosen one on your codes.

In [ ]:
WORDINGS = {
    "bare": ["a photo", "a photo of a protest"],
    "events": ["a photo of a street festival or celebration", "a photo of a political protest"],
    "descriptive": ["people gathered in a city for a party or holiday",
                    "people demonstrating in the street with banners and signs"],
    "news style": ["a news photo of a celebration", "a news photo of a demonstration"],
    "negated": ["a photo that does not show a protest", "a photo that shows a protest"],
}

train_emb = embed_images(train_table["local_path"].tolist())
test_emb = embed_images(test_table["local_path"].tolist())

def zero_shot(image_emb, prompts):
    """Probabilities for each prompt, CLIP style: softmax over 100 x cosine similarity."""
    logits = 100 * image_emb @ embed_texts(prompts).T
    return torch.tensor(logits).softmax(dim=1).numpy()

rows = []
for name, prompts in WORDINGS.items():
    pred = zero_shot(train_emb, prompts).argmax(axis=1)
    rows.append({"wording": name,
                 "balanced accuracy (training images, category labels)": balanced_accuracy_score(train_table["category_label"], pred),
                 "share predicted protest": pred.mean()})
wording_table = pd.DataFrame(rows).round(3)
wording_table

In [ ]:
best = wording_table.sort_values("balanced accuracy (training images, category labels)").iloc[-1]["wording"]
print("chosen on the training images:", best, WORDINGS[best])
test_pred = zero_shot(test_emb, WORDINGS[best]).argmax(axis=1)
print(course_data.log_score("CLIP zero-shot, prompt chosen on training data", test_table["protest"], test_pred, module="09"))

Look at how far apart the wordings are. Changing a few words can move balanced accuracy by ten
points or more, and it often changes the **share** of images labeled protest even more, which
matters if that share is your outcome. The "negated" wording usually does badly: CLIP's text
encoder is poor at negation and mostly registers that the word *protest* appears in both sentences.

Two habits help. Choose wordings on data you don't evaluate on, as we did. And report the spread
across reasonable wordings, not just the winner, so readers can see how fragile the result is.

> **✏️ Exercise 1**
>
> A **prompt ensemble** averages several wordings so that no single one dominates. For each category,
> embed the sentences you get from the templates below, average them, normalize the average to length
> 1, and classify by similarity to the two averages. Score the ensemble on the training images and on
> your codes, and log it as `"CLIP zero-shot, prompt ensemble"`.
>
> ```python
> TEMPLATES = ["a photo of {}.", "a news photo of {}.", "a blurry photo of {}.", "a photo showing {}.", "{}"]
> CATEGORIES = ["a celebration or festival", "a political protest or demonstration"]
> ```

In [ ]:
# ✅ Solution
TEMPLATES = ["a photo of {}.", "a news photo of {}.", "a blurry photo of {}.", "a photo showing {}.", "{}"]
CATEGORIES = ["a celebration or festival", "a political protest or demonstration"]

def category_vector(category):
    v = embed_texts([t.format(category) for t in TEMPLATES]).mean(axis=0)
    return v / np.linalg.norm(v)

class_vectors = np.stack([category_vector(c) for c in CATEGORIES])
train_pred_ens = (train_emb @ class_vectors.T).argmax(axis=1)
test_pred_ens = (test_emb @ class_vectors.T).argmax(axis=1)
print("training images:", round(balanced_accuracy_score(train_table["category_label"], train_pred_ens), 3))
print(course_data.log_score("CLIP zero-shot, prompt ensemble", test_table["protest"], test_pred_ens, module="09"))

# Ensembles rarely beat the best single wording by much, but they are rarely the worst either:
# the result depends less on one lucky or unlucky phrase. Radford et al. (2021) report gains of a
# few points on ImageNet from ensembling 80 templates.

## 3. A linear probe on CLIP embeddings

Zero-shot needs no labels, but if you have labeled training images, a linear probe on the same
embeddings uses them. This is the comparison that matters in practice: how much do labels add?

In [ ]:
probe = LogisticRegression(class_weight="balanced", max_iter=2000).fit(train_emb, train_table["category_label"])
print(course_data.log_score("linear probe, CLIP ViT-B/32", test_table["protest"], probe.predict(test_emb), module="09"))

With a few hundred training images, the probe usually beats zero-shot, often by a clear margin.
Zero-shot shines when you have no labels at all, when you need many categories quickly, or as a
first pass to find candidates for hand coding.

> **✏️ Exercise 2**
>
> SigLIP 2 is a newer image-text model, trained with a slightly different loss (each image-caption
> pair is scored on its own, instead of against the whole batch). The helpers in `course_torch.py`
> take a `model_name`, so you can swap it in:
>
> ```python
> SIGLIP = "google/siglip2-base-patch16-224"
> course_torch.clip_images(paths, model_name=SIGLIP)
> course_torch.clip_texts(sentences, model_name=SIGLIP)
> ```
>
> Embed the training and test images with SigLIP 2, run the zero-shot classification with the wording
> you chose in section 2 and the linear probe, and log both. Is the newer model better on your codes?

In [ ]:
# ✅ Solution
SIGLIP = "google/siglip2-base-patch16-224"
sig_train = course_torch.clip_images(train_table["local_path"], model_name=SIGLIP, cache="protest_train_siglip2.npz")
sig_test = course_torch.clip_images(test_table["local_path"], model_name=SIGLIP, cache="protest_test_siglip2.npz")

sig_text = course_torch.clip_texts(WORDINGS[best], model_name=SIGLIP)
sig_zero = (sig_test @ sig_text.T).argmax(axis=1)
print(course_data.log_score("SigLIP 2 zero-shot, same prompt", test_table["protest"], sig_zero, module="09"))

sig_probe = LogisticRegression(class_weight="balanced", max_iter=2000).fit(sig_train, train_table["category_label"])
print(course_data.log_score("linear probe, SigLIP 2", test_table["protest"], sig_probe.predict(sig_test), module="09"))

# SigLIP 2 usually does better than the original CLIP on standard benchmarks, and its probe is
# often a little stronger here as well. But a prompt chosen for one model is not necessarily the
# best for another, and with 40 test images a difference of a few points can be noise. The fair
# comparison would repeat the wording choice on the training images for SigLIP 2 too.

## 4. Searching a collection with words

Because sentences and images share one space, a sentence can be a search query. This is often the
quickest way into a large collection: find candidates for a category, then look at them.

In [ ]:
package_paths = packaging["local_path"].tolist()
package_emb = course_torch.clip_images(package_paths, cache="packaging_clip.npz")

def search(query, k=8):
    scores = package_emb @ embed_texts([query])[0]
    top = np.argsort(-scores)[:k]
    show_grid([package_paths[i] for i in top], [f"{scores[i]:.3f}" for i in top], n_cols=k, size=1.5)

for query in ["packaging with a cartoon character", "a cereal box that looks healthy and natural",
              "a chocolate cereal for children"]:
    print(query)
    search(query)

> **✏️ Exercise 3**
>
> You can also search with an image *and* words: take the embedding of one package, add the embedding
> of a phrase such as "with a cartoon character", normalize the sum, and search with that. Pick a
> package without a character, run this combined query and compare the result with searching for the
> image alone. When could this kind of query be useful in research?

In [ ]:
# ✅ Solution
def search_vector(vector, k=8, title=""):
    vector = vector / np.linalg.norm(vector)
    scores = package_emb @ vector
    top = np.argsort(-scores)[:k]
    print(title)
    show_grid([package_paths[i] for i in top], [f"{scores[i]:.3f}" for i in top], n_cols=k, size=1.5)

start = int(np.argmin(package_emb @ embed_texts(["packaging with a cartoon character"])[0]))
search_vector(package_emb[start], title="the image alone")
search_vector(package_emb[start] + embed_texts(["with a cartoon character"])[0], title="the image + 'with a cartoon character'")

# The image alone returns near copies of the starting package (same brand, same design). Adding the
# phrase pulls the results toward packages that resemble it *and* show a character. Useful for
# finding "the same kind of product, marketed differently": for example, cereals with a similar
# look but aimed at children, to study how a brand adapts its design for different audiences.
# The results are suggestions to look at, not measurements.

## 5. A measurement scale defined in words

In Bags, Vectors & Transformers you placed words on a line between two poles. CLIP lets you do the
same with images: describe the two ends of a concept in sentences, and score each image by how much
more similar it is to one end than to the other. Here we build a scale from "indulgent" to
"healthy-looking" packaging.

Then comes the step that makes it a measurement instead of a demo: **validation**. Ideally you
would compare the scale with human ratings of how healthy each package looks. Open Food Facts gives
us something different but interesting, the **Nutri-Score** (A is the most favorable nutritional
profile, E the least). The scale measures what the packaging *suggests*, the Nutri-Score what is
inside, so a weak relationship isn't necessarily a failure of the scale. Packaging that looks
healthier than the product is is a research topic in its own right.

In [ ]:
HEALTHY = "a wholesome, natural breakfast cereal with whole grains, a healthy choice"
INDULGENT = "a sweet, sugary breakfast cereal for children, a fun treat"

def scale(image_emb, positive, negative):
    poles = embed_texts([positive, negative])
    return image_emb @ poles[0] - image_emb @ poles[1]

packaging["healthy_look"] = scale(package_emb, HEALTHY, INDULGENT)
order = packaging.sort_values("healthy_look")
print("most indulgent-looking")
show_grid(order["local_path"].head(8), order["nutriscore_grade"].head(8), n_cols=8, size=1.5)
print("most healthy-looking")
show_grid(order["local_path"].tail(8), order["nutriscore_grade"].tail(8), n_cols=8, size=1.5)

In [ ]:
graded = packaging[packaging["nutriscore_grade"].isin(list("abcde"))].copy()
graded["grade_number"] = graded["nutriscore_grade"].map({g: i for i, g in enumerate("abcde")})
rho, p = spearmanr(graded["healthy_look"], graded["grade_number"])
print(f"{len(graded)} products with a Nutri-Score; Spearman correlation with the scale: {rho:.2f} (p = {p:.3f})")

graded.boxplot(column="healthy_look", by="nutriscore_grade", figsize=(5, 3.2))
plt.suptitle("")
plt.title("CLIP 'healthy-looking' score by Nutri-Score")
plt.xlabel("Nutri-Score (a = most favorable)")
plt.show()

If the scale tracks nutrition at all, the correlation is negative (healthier-looking packages get
better grades, and *a* is coded as 0). Expect a modest relationship at best. The interesting cases
are the off-diagonal ones: packages that look healthy but score D or E, and plain-looking packages
that score A.

> **✏️ Exercise 4**
>
> The scale depends on how you word its poles. Build two more versions with different sentences for
> the same two ends (for example, shorter ones like "healthy cereal" versus "sugary cereal", or ones
> that mention design, like "packaging with natural colors and images of grain" versus "packaging
> with bright colors and cartoons"). How strongly do the three versions correlate with each other,
> and with the Nutri-Score? What would you report?

In [ ]:
# ✅ Solution
VERSIONS = {
    "original": (HEALTHY, INDULGENT),
    "short": ("healthy cereal", "sugary cereal"),
    "design": ("packaging with natural colors and images of grain and fruit",
               "packaging with bright colors, cartoons and chocolate"),
}
for name, (pos, neg) in VERSIONS.items():
    packaging[f"scale_{name}"] = scale(package_emb, pos, neg)

cols = [f"scale_{n}" for n in VERSIONS]
print("correlations between versions (Spearman):")
print(packaging[cols].corr(method="spearman").round(2))
graded = packaging[packaging["nutriscore_grade"].isin(list("abcde"))].copy()
graded["grade_number"] = graded["nutriscore_grade"].map({g: i for i, g in enumerate("abcde")})
for c in cols:
    print(f"{c:15s} vs Nutri-Score: rho = {spearmanr(graded[c], graded['grade_number'])[0]:.2f}")

# The versions usually correlate well with each other but far from perfectly, so individual
# packages can move a lot depending on the wording. Report the wording you chose and why, show the
# results for alternatives, and validate the version you use against human ratings of the concept
# you mean ("looks healthy"), not only against a related but different criterion (Nutri-Score).

## 6. What CLIP learned about people

CLIP learned from captions written by people on the web, with all their habits about who does
what. The original paper already found that, tested on FairFace faces, some groups were more often
put into crime-related or non-human categories (Radford et al., 2021, section 7). Here we run a
small audit with a useful property: we know the right answer. Every person in the Congress
portraits is a politician. So we ask CLIP which occupation fits each portrait best, and check
whether it says "politician" equally often for women and men (gender from the public
congress-legislators metadata, as in module 07).

In [ ]:
OCCUPATIONS = ["a politician", "a lawyer", "a businessperson", "a teacher", "a nurse", "a doctor",
               "a news anchor", "a secretary", "a professor", "a police officer"]
portrait_emb = course_torch.clip_images(portraits["local_path"], cache="portraits_clip.npz")

def occupation_labels(prompts):
    sims = portrait_emb @ embed_texts([f"a photo of {o}" for o in prompts]).T
    return np.array(prompts)[sims.argmax(axis=1)]

portraits["clip_occupation"] = occupation_labels(OCCUPATIONS)
print(pd.crosstab(portraits["clip_occupation"], portraits["gender"], normalize="columns").round(2))

In [ ]:
def bootstrap_share(values, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    boots = [rng.choice(values, len(values)).mean() for _ in range(n_boot)]
    return values.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

rows = []
for g, group in portraits.groupby("gender"):
    share, low, high = bootstrap_share(group["clip_occupation"] == "a politician")
    rows.append({"gender": g, "n": len(group), "labeled politician": share, "low": low, "high": high})
pd.DataFrame(rows).round(3)

Read the table in two steps. First, how often is the right answer chosen at all? Second, is that
share different for women and men, beyond what the intervals allow for, and which wrong answers
does each group get? In audits like this (compare Schwemmer et al., 2020, in module 07), women in
professional roles are often given occupations that are stereotypically female, such as *nurse* or
*secretary*, more often than men are.

> **✏️ Exercise 5**
>
> An audit result can depend on the label set as much as on the model. Rerun the audit twice: once
> with `"a member of Congress"` added to the occupations, and once with `"a nurse"` and
> `"a secretary"` removed. How do the overall share correct and the gap between women and men change?

In [ ]:
# ✅ Solution
def audit(prompts, correct):
    labels = occupation_labels(prompts)
    out = {}
    for g in ["F", "M"]:
        mask = (portraits["gender"] == g).to_numpy()
        out[g] = np.isin(labels[mask], correct).mean()
    out["gap (M - F)"] = out["M"] - out["F"]
    return out

results = {
    "original list": audit(OCCUPATIONS, ["a politician"]),
    "+ member of Congress": audit(OCCUPATIONS + ["a member of Congress"], ["a politician", "a member of Congress"]),
    "- nurse, secretary": audit([o for o in OCCUPATIONS if o not in ("a nurse", "a secretary")], ["a politician"]),
}
print(pd.DataFrame(results).T.round(3))

# Adding a more specific correct label usually raises the share correct for everyone, and can
# shrink or widen the gap. Removing the stereotyped alternatives often shrinks the gap, because
# the women who were labeled nurse or secretary now get another label, sometimes the right one.
# So the size of a measured bias depends on the label set the researcher offers. Report the full
# label set, and check that your conclusion holds across reasonable alternatives.

> **✏️ Exercise 6**
>
> A colleague wants to use CLIP zero-shot to code the occupational roles of people shown in news
> photos (politician, expert, citizen, celebrity...). Using what you saw in this notebook, list the
> checks they should run before trusting the results.

**✅ One possible answer**

- **Validate against hand codes**, on a sample drawn from their own photos, with error rates per
  group (gender, and other attributes that matter for the study), as in module 07.
- **Choose prompts on held-out data**, and report the spread of results across reasonable wordings
  and label sets, not just the best one.
- **Check the label set**: missing categories force wrong answers, and stereotyped alternatives
  can create gaps between groups that come from the design as much as from the model.
- **Consider context over appearance**: an occupational role in a news photo is shown by the
  setting, captions and clothing more than by the face. If the model relies on appearance, it
  will reproduce web stereotypes; coding the context instead (podium, studio, street) may be both
  more valid and less biased.
- **Use labels where possible**: if a few hundred hand-coded images are available, a linear probe
  on the same embeddings is usually more accurate than zero-shot, and its errors can be audited
  the same way.

In [ ]:
course_data.keep_my_files()
course_data.show_scores()